In [1]:
import os
import cv2
import numpy as np

In [3]:
# =========================================================
# ILLEGAL MINING PROJECT - PATCH EXTRACTION (SITE + YEAR)
# =========================================================

import os
import cv2
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------

PATCH_SIZE = 64  # 64x64 patches

# Each entry: (site_folder, site_name)
SITES = [
    ("jodi mining",   "jodi_mining_1"),
    ("jodi mining 2", "jodi_mining_2"),
    ("jodi mining 3", "jodi_mining_3"),
]

# Root folder where patches will be saved
DATASET_ROOT = "patch_dataset"

os.makedirs(DATASET_ROOT, exist_ok=True)

# ---------------------------------------------------------
# PATCH EXTRACTION FUNCTION
# ---------------------------------------------------------

def extract_patches(image, patch_size):
    h, w, c = image.shape
    patches = []

    for y in range(0, h - patch_size + 1, patch_size):
        for x in range(0, w - patch_size + 1, patch_size):
            patch = image[y:y + patch_size, x:x + patch_size]
            patches.append(patch)

    return patches

# ---------------------------------------------------------
# MAIN PROCESSING LOOP (SITE → YEAR → IMAGE)
# ---------------------------------------------------------

total_patches_all_sites = 0

for site_dir, site_name in SITES:
    print("\n===================================")
    print(f"Processing site: {site_name}")
    print("===================================")

    site_patch_count = 0

    # Loop over YEAR folders (e.g. 2021, 2022, ...)
    for year in sorted(os.listdir(site_dir)):
        year_path = os.path.join(site_dir, year)

        if not os.path.isdir(year_path):
            continue

        print(f"\n  Year: {year}")

        # Output directory: patch_dataset/site/year/
        output_dir = os.path.join(DATASET_ROOT, site_name, year)
        os.makedirs(output_dir, exist_ok=True)

        for img_name in tqdm(os.listdir(year_path)):
            if not img_name.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
                continue

            img_path = os.path.join(year_path, img_name)
            image = cv2.imread(img_path)

            if image is None:
                print(f"Skipping unreadable image: {img_name}")
                continue

            patches = extract_patches(image, PATCH_SIZE)

            for i, patch in enumerate(patches):
                patch_filename = f"{site_name}_{year}_{img_name.split('.')[0]}_p{i}.png"
                patch_path = os.path.join(output_dir, patch_filename)
                cv2.imwrite(patch_path, patch)
                site_patch_count += 1

    total_patches_all_sites += site_patch_count

    print(f"\nTotal patches created for {site_name}: {site_patch_count}")

# ---------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------

print("\n===================================")
print("PATCH EXTRACTION COMPLETE")
print("===================================")
print(f"Total patches from all sites: {total_patches_all_sites}")
print(f"Patches saved under: {DATASET_ROOT}")


Processing site: jodi_mining_1

  Year: 2021


100%|██████████| 6/6 [00:01<00:00,  4.28it/s]



  Year: 2022


100%|██████████| 6/6 [00:01<00:00,  5.96it/s]



  Year: 2023


...


  Year: 2024


100%|██████████| 4/4 [00:00<00:00,  4.14it/s]



  Year: 2025


100%|██████████| 5/5 [00:00<00:00,  6.04it/s]


Total patches created for jodi_mining_3: 4189

PATCH EXTRACTION COMPLETE
Total patches from all sites: 11612
Patches saved under: patch_dataset


In [ ]:
import os
import random
import shutil

# -----------------------------
# USER SETTINGS (EDIT THESE)
# -----------------------------

SOURCE_FOLDER = "patch_dataset/jodi_mining_3/2025"   # year folder
DEST_FOLDER = "sampled_patches/jodi_mining_3/2025"  # where selected patches go
NUM_PATCHES = 200                                   # how many to pick

# -----------------------------
# CREATE DESTINATION FOLDER
# -----------------------------

os.makedirs(DEST_FOLDER, exist_ok=True)

# -----------------------------
# GET ALL IMAGE FILES
# -----------------------------

image_files = [
    f for f in os.listdir(SOURCE_FOLDER)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

if len(image_files) < NUM_PATCHES:
    raise ValueError(
        f"Not enough patches in folder. Found {len(image_files)}, need {NUM_PATCHES}"
    )

# -----------------------------
# RANDOMLY SELECT PATCHES
# -----------------------------

selected_files = random.sample(image_files, NUM_PATCHES)

# -----------------------------
# COPY SELECTED PATCHES
# -----------------------------

for file_name in selected_files:
    src_path = os.path.join(SOURCE_FOLDER, file_name)
    dst_path = os.path.join(DEST_FOLDER, file_name)
    shutil.copy(src_path, dst_path)

print(f"Copied {NUM_PATCHES} patches to {DEST_FOLDER}")

Copied 200 patches to sampled_patches/jodi_mining_3/2025


In [ ]:
import os
import cv2
import shutil
import numpy as np

# -------------------------------------------------
# USER PATHS (MATCHING YOUR EXACT STRUCTURE)
# -------------------------------------------------

SOURCE_FOLDER = r"sampled_patches\jodi_mining_1\2021"
NON_MINING_FOLDER = r"cnn dataset\non mining"

os.makedirs(NON_MINING_FOLDER, exist_ok=True)

# -------------------------------------------------
# THRESHOLDS (SAFE DEFAULTS)
# -------------------------------------------------

WHITE_THRESHOLD = 240      # cloud
BLACK_THRESHOLD = 15       # no data
GREEN_DOMINANCE_RATIO = 1.3  # dense vegetation

# -------------------------------------------------
# CHECK FUNCTION
# -------------------------------------------------

def is_obvious_non_mining(img):
    if img is None:
        return True

    mean_intensity = img.mean()

    # White / cloud
    if mean_intensity > WHITE_THRESHOLD:
        return True

    # Black / no data
    if mean_intensity < BLACK_THRESHOLD:
        return True

    # Green dominance (forest)
    b, g, r = cv2.split(img)
    green_ratio = g.mean() / ((r.mean() + b.mean()) / 2 + 1e-5)

    if green_ratio > GREEN_DOMINANCE_RATIO:
        return True

    return False

# -------------------------------------------------
# MAIN LOOP
# -------------------------------------------------

moved = 0
kept = 0

for file in os.listdir(SOURCE_FOLDER):
    if not file.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    src_path = os.path.join(SOURCE_FOLDER, file)
    img = cv2.imread(src_path)

    if is_obvious_non_mining(img):
        dst_path = os.path.join(NON_MINING_FOLDER, file)
        shutil.move(src_path, dst_path)
        moved += 1
    else:
        kept += 1

print("===================================")
print("AUTO NON-MINING SEPARATION DONE")
print("===================================")
print("Moved to non-mining folder :", moved)
print("Remaining for manual label:", kept)

AUTO NON-MINING SEPARATION DONE
Moved to non-mining folder : 78
Remaining for manual label: 122
